# Modelos de regresión en SKlearn
## 1. Objetivo

Familiarizarse con los conceptos de modelos de regresion

## Datos de Melb Houses
* Suburb: Suburb
* Address: Address
* Rooms: Number of rooms
* Price: Price in Australian dollars
* Method:
  * S - property sold;
  * SP - property sold prior;
  * PI - property passed in;
  * PN - sold prior not disclosed;
  * SN - sold not disclosed;
  * NB - no bid;
  * VB - vendor bid;
  * W - withdrawn prior to auction;
  * SA - sold after auction;
  * SS - sold after auction price not disclosed.
  * N/A - price or highest bid not available.

* Type:
  * br - bedroom(s);
  * h - house,cottage,villa, semi,terrace;
  * u - unit, duplex;
  * t - townhouse;
  * dev site - development site;
  * res - other residential.

* SellerG: Real Estate Agent

* Date: Date sold

* Distance: Distance from CBD in Kilometres
* Regionname: General Region (West, North West, North, North east …etc)
* Propertycount: Number of properties that exist in the suburb.
* Bedroom2 : Scraped # of Bedrooms (from different source)
* Bathroom: Number of Bathrooms
* Car: Number of carspots
* Landsize: Land Size in Metres
* BuildingArea: Building Size in Metres
* YearBuilt: Year the house was built
* CouncilArea: Governing council for the area
* Lattitude: Self explanitory
* Longtitude: Self explanitory


* Melb Data: https://www.kaggle.com/code/dansbecker/handling-missing-values/data?select=melb_data.csv
* Sklearn Cheat Sheet from Datacamp: https://media.datacamp.com/legacy/image/upload/v1676302389/Marketing/Blog/Scikit-Learn_Cheat_Sheet.pdf

## 2. Librerias de trabajo

In [1]:
# Instala libreria Pandas si no la tenemos
#pip install pandas seaborn scikit-learn -y

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    ElasticNet
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    LabelBinarizer,
    OneHotEncoder
)

from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    ElasticNet
)

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## 3. Lectura de datos

Primero nos encargaremos de leer los datos, indicando a Python donde se encuentra la carpeta que contiene los datos y los nombres de los archivos relevantes para el análisis.

In [3]:
#  Indicamos la ruta a la carpeta de de tu computadora 
# donde se ubican los datos del E-commerce
# Ejemplo: "C:\Usuarios\[tu nombre]\Descargas"

DATA_PATH="/Users/cesar/sandbox/ai_programming_foundations/data"

Ahora procederemos a definir una variable que indique el nombre del archivo junto con su extensión (por ejemplo, `.csv`):

In [4]:
FILE_DATA_PATH = "melb_data.csv"

Echaremos mano de la utilidad `os.path.join` de Python que indicar rutas en tu computadora donde se ubican archivos, así Pandas encontrá los archivos de datos.


**Ejemplo**

A continuación mostraremos un ejemplo leyendo el archivo `melb_data.csv`.
csv`:

In [5]:
# Ejemplo
print(f"Ruta del archivo: {FILE_DATA_PATH}")
print(os.path.join(DATA_PATH, FILE_DATA_PATH))

Ruta del archivo: melb_data.csv
/Users/cesar/sandbox/ai_programming_foundations/data/melb_data.csv


In [6]:
# Leemos con pandas
df = pd.read_csv(
    os.path.join(DATA_PATH, FILE_DATA_PATH)
    )

In [7]:
df.sample(10)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
2728,Footscray,23 Moore St,2,h,800000,S,Village,27/06/16,6.4,3011,...,1,2.0,260,NaN,1940.0,Maribyrnong,-37.79690,144.90420,Western Metropolitan,7570
3012,Hampton,1/30 Grenville St,2,u,895000,S,hockingstuart,03/09/16,13.7,3188,...,1,1.0,228,87.0,1975.0,Bayside,-37.93550,145.00080,Southern Metropolitan,5454
6298,Thornbury,314A Gillies St,2,h,874000,S,Love,19/11/16,6.5,3071,...,1,2.0,333,2.0,1985.0,Darebin,-37.76420,145.01920,Northern Metropolitan,8870
4553,Oakleigh South,13 Farm Rd,3,h,937500,S,Ray,12/11/16,14.7,3167,...,1,2.0,530,NaN,NaN,Monash,-37.92450,145.09370,South-Eastern Metropolitan,3692
7861,Flemington,75 Farnham St,3,h,1205000,S,Nelson,22/04/17,4.4,3031,...,1,0.0,251,NaN,NaN,Moonee Valley,-37.78230,144.93170,Northern Metropolitan,3593
5937,Sunshine,36 Matthews St,5,h,895000,PI,Barry,07/05/16,12.6,3020,...,2,2.0,690,NaN,NaN,Brimbank,-37.79330,144.84080,Western Metropolitan,3755
4247,Newport,15 Jubilee St,4,h,1040000,S,Greg,17/09/16,8.4,3015,...,2,2.0,299,198.0,2004.0,Hobsons Bay,-37.84770,144.87060,Western Metropolitan,5498
11167,Prahran,11A Percy St,2,h,1310000,S,hockingstuart,12/08/17,4.6,3181,...,2,1.0,78,NaN,NaN,Stonnington,-37.85083,144.99092,Southern Metropolitan,7717
389,Ashburton,3/75 Victory Bvd,3,t,968000,S,Jellis,18/06/16,11.0,3147,...,1,1.0,227,NaN,NaN,Boroondara,-37.87090,145.08570,Southern Metropolitan,3052
12193,Wantirna South,41 Witken Av,4,h,930000,S,Noel,29/07/17,14.7,3152,...,2,2.0,639,NaN,NaN,Knox,-37.87890,145.24262,Eastern Metropolitan,7082


Revisemo las información de los datos:

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  int64  
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  int64  
 10  Bedroom2       13580 non-null  int64  
 11  Bathroom       13580 non-null  int64  
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  int64  
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

## 2. Modelos Básicos de Aprendizaje de Máquina

Esta sección aborda algunos de los principales modelos usados en problemas de aprendizaje supervisado.

### 2.1 Modelos de Regresión Lineal

Los modelos de regresión son modelos estadísticos que asumen la existencia de una relación lineal entre las características del problema y la variable a predecir, es decir, la variable objetivo se puede describir de la forma

$$y_i = \beta_0 + \beta_1 x_{i1}  + \beta_2 x_{i2} + \ldots + \beta_n x_{in}+ \epsilon_i$$

Donde los coeficientes $\beta_i$ son parámetros por determinarse y $\epsilon_i$ es una representación del ruido en la relación entre $y_i$ y $x_{i1}, \ldots, x_{in}$. En la jerga estadística los variables reciben el nombre de regresores.

Debe notarse que dado que los parámetros deben estimarse existen supuestos de tipo estadístico para que la relación lineal entre las variables y el objetivo se aproxime de buena manera, es decir, el residuno, nombre que recib el error entre $y$ y la $x_{i1}, \ldots, x_{1n}$, que generalmente se describe en termino del error cuadrático medio.

La teoría estadística señala algunos de los supuestos que deben cumplirse para que la regresión sea un modelo con buen funcionamiento (es decir controlando el sesgo y la varianza de la predicción):

* **Linealidad:** La relación entre la variable dependiente $Y$ y las variables independientes $X$ debe ser lineal.
* **Normalidad de los residuos:** Los residuos deben seguir una distribución normal, este supuesto se puede relajar a una distribución aproximadamente normal.
* **Homogeneidad de la varianza de los residuos:** Los residuos deben tener una varianza constante (homocedasticidad), es decir se espera que la variación de los residuos no se dispare en alguna región del espacio.
* **Independencia de los residuos:** Los residuos deben ser independientes los unos de los otros, es decir  no deben estar correlacionados entre sí.

Por otro lado la estimación de los coeficientes $\beta_i$ se realiza con técnicas de análisis numérico y que siguen principios estadísticos que, cumpliendose los supuestos anteriores, aseguran que el modelo obtenido tenga la combinación de coeficientes que asegura una varianza mínimo (Teorema de Gauss-Markov). Normalmente los paquetes de cómputo científico realizan los cálculos correspondientes, pero la verificación de dichos supuestos es responsabilidad de quien realiza el análisis.

Es dable mencionar que los modelos de regresión son herramientas poderosas de aprendizaje supervisado, pero tienen algunos puntos a considerar:

* Se ven afectados fuertemente por la presencia de valores atípicos y la presencia de escala distintas en los datos, por lo que es común pre-procesar los datos para eliminar valores ruidosos y asegurando que todos sus componentes tengan órdenes de magnitud comparables.
* Requieren que los variables dentro de los mismos sean linealmente independientes en el sentido del álgebra lineal, normalmente un buen análisis de correlación puede ayudar a desechar aquellas variables entre las que exista correlación lineal para mejorar su desempeño
* Son muy flexibles, típicamente puede transformarse para capturar relaciones no lineales.
* Generalmente la cantidad de variables que se puede incluir está limitada por la cantidad de puntos en el conjunto de entrenamiento.
* Si todas las variables tienen la misma escala, los coeficientes se pueden interpretar como el efecto que tiene una característica sobre la variable objetivo, considerando que todas las demás características se quedan fija (*ceteris paribus*).

#### 2.1.1 Modelos de Regresión Lineal y Regularización

Para mejorar evitar el sobreajuste de estos modelos, existen técnicas que penalizan la complejidad del modelo, es decir el valor que pueden tomar los coeficientes y que en general inducen mejores resultados para predecir.

Los modelos de regresión que incluyen penalizaciones sobre los coeficientes tienen denominaciones especiales:

* **Lasso:** Este modelo es similar al descrito arriba pero considerando que los coefientes se encuentre en una región definida por $\sum |\beta_i| \leq K_1$ donde $K_1$ es alguna constante. Esta métrica generalmente hace que algunos de los coeficientes $\beta_i$ sean cercanos a cero o bien se anulen.
* **Ridge:** Similar al modelo previo, pero considerando que los coefientes se encuentre en una región definida por $\sum |\beta_i|^2 \leq K_2$ donde $K_2$ es alguna constante.
* **Elastic Net:** En este caso, se pide que los coeficientes satisfagan la restricción $\sum |\beta_i| + \sum |\beta_i|^2 \leq K_3$ donde $K_3$ es cierta constante.


Una forma alternativa de describir lo anterior, es la formulación de Lagrange los problemas de minimización con restriciones, donde esencialmente se busca minimizar la expresión para encontrar el error más el valor de los parámetros

* **Lasso:** $$||y - \beta X||^2_2 + \alpha ||\beta||^2_2$$ donde el coeficiente $\alpha$ es un parámetros que controla la restricción del tamaño de $\beta$. 
* **Ridge:** $$||y - \beta X||^2_2 + \alpha ||\beta||^2_1$$ donde el coeficiente $\alpha$ es un parámetros que controla la restricción del tamaño de $\beta$. 
* **Elastic NEt:** $$||y - \beta X||^2_2 + \alpha_1 ||\beta||^2_1 + \alpha_2  ||\beta||^2_2$$ donde el coeficiente $\alpha$ es un parámetros que controla la restricción del tamaño de $\beta$. (La parametrización de Sklear es equivalente, ver https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html#sklearn.linear_model.ElasticNet)

En Sklearn, dichos modelos se encuentran disponibles en las clases `LinearRegression`, `Lasso`, `Ridge` y `ElasticNet`. En Python, los valores de $\alpha_i$ y sus reparamtrizaciónes se utilizan para controlar que tantos se restringe a los coeficientes para controlar el sobreajuste. Al ser parámetros que no dependen específicamente de los datos, de denominar **hiper-parámetros**.

# 3. Modelos de regresión para predecir el precio de Melb Houses

In [9]:
# Define listas de columnas que van a emplearse en el modelado
num_features = [
    'Rooms', 
    'BuildingArea',
    'Landsize',
    'Distance',
    'Bathroom',
    'YearBuilt'
 ]

cat_cols = ['Regionname', 'Type']

# Lista que tiene todas los grupos de columnas
non_target_cols = num_features + cat_cols

target = ['Price']

In [10]:
df["BuildingArea"] = df["BuildingArea"].fillna(0)

In [11]:
df["YearBuilt"] = df["YearBuilt"].fillna(df["YearBuilt"].median())

In [12]:
df[non_target_cols].isna().sum()

Rooms           0
BuildingArea    0
Landsize        0
Distance        0
Bathroom        0
YearBuilt       0
Regionname      0
Type            0
dtype: int64

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    df[non_target_cols],
    df[target],
    test_size=0.2,
)

In [14]:
# Pipeline para escalar con estandar z-score
numerical_pipe = Pipeline([
    ('standar_scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

In [15]:
# Combina ambos procesos en columnas espeficadas en listas
pre_processor = ColumnTransformer([
    ('numerical', numerical_pipe, num_features),
    ('categorical', categorical_pipe, cat_cols),
], remainder='passthrough')

In [16]:
models_regresion = {
    'regression': LinearRegression(),
    'lasso_01': Lasso(alpha=0.1),
    'lasso_001': Lasso(alpha=0.01),
    'lasso_0001': Lasso(alpha=0.001),
    'lasso_000001': Lasso(alpha=0.0001),
    'ridge_01': Ridge(alpha=0.1),
    'ridge_001': Ridge(alpha=0.01),
    'ridge_0001': Ridge(alpha=0.001),
    'ridge_000001': Ridge(alpha=0.0001),
    'elastic_05_03': ElasticNet(alpha=0.5, l1_ratio=0.3),
    'elastic_0001_01': ElasticNet(alpha=0.0001, l1_ratio=0.0001)
    }

In [17]:
models = []
models_train_errors = []
models_test_errors = []

for model_name in models_regresion.keys():

    print("Modelo:", model_name)
    model = Pipeline([
        ('transform', pre_processor),
        ('model', models_regresion[model_name])
    ])

    # Ajusta el modelo con los datos de prueba
    model.fit(X_train[non_target_cols],y_train)

    y_train_pred = model.predict(X_train[non_target_cols])
    y_test_pred = model.predict(X_test[non_target_cols])

    # error en conjunto de entrenamiento y prueba
    error_train = root_mean_squared_error(y_train, y_train_pred)
    error_test = root_mean_squared_error(y_test, y_test_pred)

    # errores
    print("Error RSME en train:", round(error_train,4) )
    print("Error RSME en test:", round(error_test,4) )

    print("----------------------------------------------")

    models.append(model_name)
    models_train_errors.append(error_train)
    models_test_errors.append(error_test)

    

Modelo: regression
Error RSME en train: 401809.01
Error RSME en test: 441694.1063
----------------------------------------------
Modelo: lasso_01
Error RSME en train: 401809.0101
Error RSME en test: 441694.1668
----------------------------------------------
Modelo: lasso_001
Error RSME en train: 401809.01
Error RSME en test: 441694.1124
----------------------------------------------
Modelo: lasso_0001
Error RSME en train: 401809.01
Error RSME en test: 441694.1069
----------------------------------------------
Modelo: lasso_000001
Error RSME en train: 401809.01
Error RSME en test: 441694.1064
----------------------------------------------
Modelo: ridge_01
Error RSME en train: 401809.0129
Error RSME en test: 441695.5055
----------------------------------------------
Modelo: ridge_001
Error RSME en train: 401809.0101
Error RSME en test: 441694.2463
----------------------------------------------
Modelo: ridge_0001
Error RSME en train: 401809.01
Error RSME en test: 441694.1203
-------------

In [18]:
pd.DataFrame({
    "model": models,
    "rmse_train": models_train_errors,
    "rmse_test": models_test_errors,
}).sort_values(["rmse_test"])

,model,rmse_train,rmse_test
0,regression,401809.010040,441694.106342
4,lasso_000001,401809.010040,441694.106403
3,lasso_0001,401809.010040,441694.106947
8,ridge_000001,401809.010040,441694.107742
2,lasso_001,401809.010040,441694.112387
7,ridge_0001,401809.010040,441694.120334
1,lasso_01,401809.010055,441694.166800
6,ridge_001,401809.010069,441694.246262
5,ridge_01,401809.012941,441695.505523
10,elastic_0001_01,401809.355340,441709.906680


In [19]:
models_regresion["regression"].coef_

array([[ 170872.21455155,   14099.74294483,   15299.9766413 ,
        -233926.00960457,  151984.41922096,  -62845.13775568,
         -34267.8223517 ,  234576.06920069, -263066.07411141,
          92959.72700153,  156335.38914707,  216611.34312813,
        -318566.35985208,  -84582.27216224,  258469.75814156,
         -26635.8101376 , -231833.94800396]])

In [20]:
models_regresion["lasso_001"].coef_

array([ 170872.18640115,   14099.7627419 ,   15299.97791371,
       -233925.8454407 ,  151984.44659172,  -62845.14642605,
         41125.61612433,  309966.51778422, -187672.37211378,
        168349.21983373,  231728.32303391,  292004.92426974,
       -243172.72795019,   -9185.21898931,  440112.03886253,
        155006.34249175,  -50191.58230119])

## 4. Salvando el modelo

In [25]:
import pickle
pickle.dump(model, open('my_model.pkl', 'wb'))

In [26]:
!ls *.pkl

best_model.pkl mi_model.pkl   my_model.pkl


Para volver a cargar el modelo:

In [27]:
pickled_model = pickle.load(open('my_model.pkl', 'rb'))

In [29]:
help(pickled_model)

Help on Pipeline in module sklearn.pipeline object:

class Pipeline(sklearn.utils.metaestimators._BaseComposition)
 |  Pipeline(steps, *, transform_input=None, memory=None, verbose=False)
 |
 |  A sequence of data transformers with an optional final predictor.
 |
 |  `Pipeline` allows you to sequentially apply a list of transformers to
 |  preprocess the data and, if desired, conclude the sequence with a final
 |  :term:`predictor` for predictive modeling.
 |
 |  Intermediate steps of the pipeline must be transformers, that is, they
 |  must implement `fit` and `transform` methods.
 |  The final :term:`estimator` only needs to implement `fit`.
 |  The transformers in the pipeline can be cached using ``memory`` argument.
 |
 |  The purpose of the pipeline is to assemble several steps that can be
 |  cross-validated together while setting different parameters. For this, it
 |  enables setting parameters of the various steps using their names and the
 |  parameter name separated by a `'__